In [ ]:
# Cell 1: تجهيز البيئة عبر سكريبت مستقل
from pathlib import Path
import logging
import os
import shutil
import subprocess
import sys

CELL_NAME = "cell_01_setup_script"
CELL_REV = "2026-04-19-r2"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        stderr_tail = (result.stderr or "").strip()[-2000:]
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}\n{stderr_tail}")
    return result

def resolve_venv_python(venv_dir: Path) -> Path:
    if os.name == "nt":
        return venv_dir / "Scripts" / "python.exe"
    return venv_dir / "bin" / "python"

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
REPO_URL = "https://github.com/kareemkamal10/NAMAA-Egyptian-Voice.git"
VENV_DIR = Path("/content/namaa-venv")
SETUP_SCRIPT = PROJECT_DIR / "scripts" / "setup_env.py"
RECREATE_VENV = False

logger.info("Start Cell 1")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 1 revision: {CELL_REV}")
logger.info(f"Cell 1 revision: {CELL_REV}")

try:
    if not PROJECT_DIR.exists():
        logger.info("Cloning repository")
        run_cmd(["git", "clone", REPO_URL, str(PROJECT_DIR)])
    else:
        logger.info("Repository already exists; pulling latest changes")
        run_cmd(["git", "pull", "--ff-only"], cwd=str(PROJECT_DIR))

    run_cmd(["git", "lfs", "install"], cwd=str(PROJECT_DIR))
    run_cmd(["git", "lfs", "pull"], cwd=str(PROJECT_DIR))

    if not SETUP_SCRIPT.exists():
        raise FileNotFoundError(f"Setup script not found: {SETUP_SCRIPT}")

    setup_cmd = [
        sys.executable,
        str(SETUP_SCRIPT),
        "--project-dir",
        str(PROJECT_DIR),
        "--venv-path",
        str(VENV_DIR),
    ]
    if RECREATE_VENV:
        setup_cmd.append("--recreate")

    try:
        run_cmd(setup_cmd, cwd=str(PROJECT_DIR))
    except Exception as setup_error:
        logger.warning(f"setup_env.py failed once, trying fallback: {setup_error}")
        print("setup_env.py failed once; trying portable virtualenv fallback...")

        if VENV_DIR.exists():
            shutil.rmtree(VENV_DIR)

        run_cmd([sys.executable, "-m", "pip", "install", "-U", "virtualenv"], cwd=str(PROJECT_DIR))
        run_cmd([sys.executable, "-m", "virtualenv", "--python", sys.executable, str(VENV_DIR)], cwd=str(PROJECT_DIR))

        fallback_venv_python = resolve_venv_python(VENV_DIR)
        run_cmd([str(fallback_venv_python), "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], cwd=str(PROJECT_DIR))
        run_cmd([str(fallback_venv_python), "-m", "pip", "install", "-r", "requirements.txt"], cwd=str(PROJECT_DIR))

    VENV_PYTHON = resolve_venv_python(VENV_DIR)
    if not VENV_PYTHON.exists():
        raise FileNotFoundError(f"Virtualenv python not found: {VENV_PYTHON}")

    print(f"VENV Python: {VENV_PYTHON}")
    logger.info(f"VENV_PYTHON={VENV_PYTHON}")
except Exception:
    logger.exception("Cell 1 failed")
    raise

logger.info("End Cell 1")


In [ ]:
# Cell 2: Runtime check داخل نفس virtualenv
from pathlib import Path
import logging
import os
import subprocess
import textwrap

CELL_NAME = "cell_02_runtime_check"
CELL_REV = "2026-04-19-r1"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        stderr_tail = (result.stderr or "").strip()[-1500:]
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}\n{stderr_tail}")
    return result

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
VENV_DIR = Path("/content/namaa-venv")
if os.name == "nt":
    default_venv_python = VENV_DIR / "Scripts" / "python.exe"
else:
    default_venv_python = VENV_DIR / "bin" / "python"

VENV_PYTHON = Path(globals().get("VENV_PYTHON", default_venv_python))

def run_python_script(script_content, cwd=None):
    script_dir = PROJECT_DIR / "tmp_scripts"
    script_dir.mkdir(parents=True, exist_ok=True)
    script_file = script_dir / f"{CELL_NAME}.py"
    script_file.write_text(textwrap.dedent(script_content), encoding="utf-8")

    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(PROJECT_DIR) if not existing_pythonpath else f"{PROJECT_DIR}{os.pathsep}{existing_pythonpath}"
    return run_cmd([str(VENV_PYTHON), str(script_file)], cwd=cwd, env=env)

logger.info("Start Cell 2")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 2 revision: {CELL_REV}")
logger.info(f"Cell 2 revision: {CELL_REV}")

try:
    if not VENV_PYTHON.exists():
        raise FileNotFoundError("Virtualenv python not found. Run Cell 1 first.")
    if not (PROJECT_DIR / "app.py").exists():
        raise FileNotFoundError("app.py not found in project directory. Run Cell 1 first.")

    script = """
import sys
import torch

print('Python:', sys.version.replace(chr(10), ' '))
print('Executable:', sys.executable)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
"""
    run_python_script(script, cwd=str(PROJECT_DIR))
except Exception:
    logger.exception("Cell 2 failed")
    raise

logger.info("End Cell 2")


In [ ]:
# Cell 3: إدخال النص وحفظه (جلسة واحدة)
from pathlib import Path
import logging
from IPython.display import display, Markdown
import ipywidgets as widgets

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

CELL_NAME = "cell_03_prompt_input"
CELL_REV = "2026-04-19-r1"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

logger.info("Start Cell 3")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 3 revision: {CELL_REV}")
logger.info(f"Cell 3 revision: {CELL_REV}")

SESSION_PROMPT_TEXT = ""

default_text = "انا سبت الشغل و راجع دلوقتي علي طول."
PROMPT_TEXT_WIDGET = widgets.Textarea(
    value=default_text,
    placeholder="اكتب النص الذي تريد تحويله لصوت هنا...",
    description="",
    layout=widgets.Layout(width="100%", height="120px"),
)

PROMPT_SAVE_BUTTON = widgets.Button(
    description="حفظ النص",
    button_style="success",
    icon="check",
)
PROMPT_STATUS_WIDGET = widgets.HTML("<b>❌ لم يتم حفظ النص بعد.</b>")

def _save_prompt(_):
    global SESSION_PROMPT_TEXT
    value = str(PROMPT_TEXT_WIDGET.value).strip()
    if not value:
        PROMPT_STATUS_WIDGET.value = "<b>❌ النص فارغ. اكتب النص ثم اضغط حفظ.</b>"
        logger.warning("Prompt save failed: empty text")
        return

    SESSION_PROMPT_TEXT = value
    PROMPT_STATUS_WIDGET.value = f"<b>✅ تم حفظ النص ({len(value)} حرف).</b>"
    logger.info(f"Prompt saved with length={len(value)}")

PROMPT_SAVE_BUTTON.on_click(_save_prompt)

display(Markdown("### أدخل النص المراد توليده"))
display(PROMPT_TEXT_WIDGET)
display(widgets.HBox([PROMPT_SAVE_BUTTON, PROMPT_STATUS_WIDGET]))
print("بعد الكتابة، اضغط حفظ النص وتأكد من ظهور علامة ✅.")
logger.info("Prompt widgets displayed")
logger.info("End Cell 3")


In [ ]:
# Cell 4: رفع الريفرنس الصوتي (اختياري) مع حفظ الحالة
from pathlib import Path
import logging
from IPython.display import display, Markdown
import ipywidgets as widgets

try:
    from google.colab import output, files
    output.enable_custom_widget_manager()
except Exception:
    files = None

CELL_NAME = "cell_04_optional_reference"
CELL_REV = "2026-04-19-r4"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
UPLOADS_DIR = PROJECT_DIR / "uploads"
UPLOADS_DIR.mkdir(parents=True, exist_ok=True)

SESSION_REFERENCE_PATH = ""


def _normalize_bytes(content):
    if content is None:
        return b""
    if isinstance(content, memoryview):
        return content.tobytes()
    if hasattr(content, "tobytes"):
        try:
            return content.tobytes()
        except Exception:
            pass
    if isinstance(content, bytearray):
        return bytes(content)
    if isinstance(content, bytes):
        return content
    try:
        return bytes(content)
    except Exception:
        return b""


def _extract_from_widget_value(value):
    if not value:
        return "", b""

    if isinstance(value, dict):
        entries = list(value.values())
    elif isinstance(value, (list, tuple)):
        entries = list(value)
    else:
        return "", b""

    if not entries:
        return "", b""

    entry = entries[0]
    if not isinstance(entry, dict):
        return "", b""

    file_name = entry.get("name")
    content = entry.get("content")

    metadata = entry.get("metadata") if isinstance(entry.get("metadata"), dict) else {}
    if not file_name:
        file_name = metadata.get("name", "")
    if content is None:
        content = metadata.get("content")

    content = _normalize_bytes(content)
    if not file_name or not content:
        return "", b""

    return str(file_name), content


REFERENCE_STATUS_WIDGET = widgets.HTML("<b>✅ لا يوجد ريفرنس محفوظ (اختياري).</b>")


def _save_reference_file(file_name, content, source):
    global SESSION_REFERENCE_PATH
    safe_name = Path(str(file_name or "reference_audio.wav")).name
    output_path = UPLOADS_DIR / safe_name
    output_path.write_bytes(_normalize_bytes(content))
    SESSION_REFERENCE_PATH = str(output_path)
    REFERENCE_STATUS_WIDGET.value = f"<b>✅ تم حفظ الريفرنس:</b> {output_path.name}"
    logger.info(f"Reference saved via {source}: {output_path}")


REFERENCE_UPLOAD_WIDGET = widgets.FileUpload(
    accept=".wav,.mp3,.flac,.ogg,.m4a",
    multiple=False,
    description="رفع ملف الريفرنس",
    button_style="primary",
)


def _on_widget_change(change):
    name, content = _extract_from_widget_value(change.get("new"))
    if content:
        _save_reference_file(name, content, "FileUploadWidget")


REFERENCE_UPLOAD_WIDGET.observe(_on_widget_change, names="value")

COLAB_UPLOAD_BUTTON = widgets.Button(
    description="رفع عبر نافذة Colab",
    button_style="info",
    icon="upload",
)


def _on_colab_upload_click(_):
    if files is None:
        REFERENCE_STATUS_WIDGET.value = "<b>❌ رفع Colab غير متاح في هذه البيئة.</b>"
        return
    uploaded = files.upload()
    if uploaded:
        name, content = next(iter(uploaded.items()))
        _save_reference_file(name, content, "ColabUploadButton")
    else:
        REFERENCE_STATUS_WIDGET.value = "<b>❌ لم يتم اختيار ملف.</b>"


COLAB_UPLOAD_BUTTON.on_click(_on_colab_upload_click)

CLEAR_REFERENCE_BUTTON = widgets.Button(
    description="تخطي الريفرنس",
    button_style="warning",
    icon="times",
)


def _clear_reference(_):
    global SESSION_REFERENCE_PATH
    SESSION_REFERENCE_PATH = ""
    REFERENCE_STATUS_WIDGET.value = "<b>✅ تم حفظ الحالة: بدون ريفرنس.</b>"
    logger.info("Reference cleared by user")


CLEAR_REFERENCE_BUTTON.on_click(_clear_reference)

logger.info("Start Cell 4")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 4 revision: {CELL_REV}")
logger.info(f"Cell 4 revision: {CELL_REV}")

display(Markdown("### (اختياري) ارفع ملف الريفرنس الصوتي"))
controls = [REFERENCE_UPLOAD_WIDGET, CLEAR_REFERENCE_BUTTON]
if files is not None:
    controls.insert(1, COLAB_UPLOAD_BUTTON)

display(widgets.VBox(controls + [REFERENCE_STATUS_WIDGET]))
print("ارفع الملف أو اضغط تخطي الريفرنس، وتأكد من ظهور علامة ✅.")
logger.info("Reference upload controls displayed")
logger.info("End Cell 4")


In [ ]:
# Cell 5: إعدادات التوليد + outputname مع حفظ الحالة
from pathlib import Path
import logging
from IPython.display import display, Markdown
import ipywidgets as widgets

CELL_NAME = "cell_05_settings"
CELL_REV = "2026-04-19-r1"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

SESSION_EXAGGERATION = 0.5
SESSION_CFG_PACE = 0.5
SESSION_SEED = 0
SESSION_TEMPERATURE = 0.8
SESSION_OUTPUT_NAME = "output"

EXAGGERATION_WIDGET = widgets.BoundedFloatText(
    value=0.5,
    min=0.25,
    max=2.0,
    step=0.05,
    description="EXAGGERATION",
    layout=widgets.Layout(width="340px"),
)
CFG_PACE_WIDGET = widgets.BoundedFloatText(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CFG_PACE",
    layout=widgets.Layout(width="340px"),
)
SEED_WIDGET = widgets.BoundedIntText(
    value=0,
    min=0,
    step=1,
    description="SEED",
    layout=widgets.Layout(width="340px"),
)
TEMPERATURE_WIDGET = widgets.BoundedFloatText(
    value=0.8,
    min=0.05,
    max=5.0,
    step=0.05,
    description="TEMPERATURE",
    layout=widgets.Layout(width="340px"),
)
OUTPUT_NAME_WIDGET = widgets.Text(
    value="output",
    description="outputname",
    placeholder="مثال: output أو my_run",
    layout=widgets.Layout(width="340px"),
)

SETTINGS_SAVE_BUTTON = widgets.Button(
    description="حفظ الإعدادات",
    button_style="success",
    icon="check",
)
SETTINGS_STATUS_WIDGET = widgets.HTML("<b>❌ لم يتم حفظ الإعدادات بعد.</b>")


def _save_settings(_):
    global SESSION_EXAGGERATION, SESSION_CFG_PACE, SESSION_SEED, SESSION_TEMPERATURE, SESSION_OUTPUT_NAME

    SESSION_EXAGGERATION = float(EXAGGERATION_WIDGET.value)
    SESSION_CFG_PACE = float(CFG_PACE_WIDGET.value)
    SESSION_SEED = int(SEED_WIDGET.value)
    SESSION_TEMPERATURE = float(TEMPERATURE_WIDGET.value)

    output_value = str(OUTPUT_NAME_WIDGET.value).strip()
    SESSION_OUTPUT_NAME = output_value or "output"

    SETTINGS_STATUS_WIDGET.value = (
        "<b>✅ تم حفظ الإعدادات.</b> "
        f"EXAGGERATION={SESSION_EXAGGERATION}, "
        f"CFG_PACE={SESSION_CFG_PACE}, "
        f"SEED={SESSION_SEED}, "
        f"TEMPERATURE={SESSION_TEMPERATURE}, "
        f"outputname={SESSION_OUTPUT_NAME}"
    )
    logger.info(
        "Settings saved: "
        f"EXAGGERATION={SESSION_EXAGGERATION}, "
        f"CFG_PACE={SESSION_CFG_PACE}, "
        f"SEED={SESSION_SEED}, "
        f"TEMPERATURE={SESSION_TEMPERATURE}, "
        f"outputname={SESSION_OUTPUT_NAME}"
    )


SETTINGS_SAVE_BUTTON.on_click(_save_settings)

logger.info("Start Cell 5")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 5 revision: {CELL_REV}")
logger.info(f"Cell 5 revision: {CELL_REV}")

display(Markdown("### إعدادات التوليد (اختيارية)"))
display(EXAGGERATION_WIDGET)
display(CFG_PACE_WIDGET)
display(SEED_WIDGET)
display(TEMPERATURE_WIDGET)
display(OUTPUT_NAME_WIDGET)
display(widgets.HBox([SETTINGS_SAVE_BUTTON, SETTINGS_STATUS_WIDGET]))
print("اضغط حفظ الإعدادات وتأكد من ظهور علامة ✅.")
logger.info("Settings widgets displayed")
logger.info("End Cell 5")


In [ ]:
# Cell 6: تشغيل CLI للجلسة الحالية + معاينة الناتج
from pathlib import Path
import logging
import os
import re
import subprocess
from IPython.display import Audio, display

CELL_NAME = "cell_06_run_cli"
CELL_REV = "2026-04-19-r1"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)


def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        stderr_tail = (result.stderr or "").strip()[-2500:]
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}\n{stderr_tail}")
    return result


PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
VENV_DIR = Path("/content/namaa-venv")
if os.name == "nt":
    default_venv_python = VENV_DIR / "Scripts" / "python.exe"
else:
    default_venv_python = VENV_DIR / "bin" / "python"

VENV_PYTHON = Path(globals().get("VENV_PYTHON", default_venv_python))

logger.info("Start Cell 6")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 6 revision: {CELL_REV}")
logger.info(f"Cell 6 revision: {CELL_REV}")

try:
    if not VENV_PYTHON.exists():
        raise FileNotFoundError("Virtualenv python not found. Run Cell 1 first.")
    if not (PROJECT_DIR / "app.py").exists():
        raise FileNotFoundError("app.py not found in project directory. Run Cell 1 first.")

    prompt_value = str(globals().get("SESSION_PROMPT_TEXT", "")).strip()
    if not prompt_value:
        prompt_widget = globals().get("PROMPT_TEXT_WIDGET")
        if prompt_widget is not None:
            prompt_value = str(prompt_widget.value).strip()

    if not prompt_value:
        raise ValueError("الرجاء كتابة النص في Cell 3 ثم حفظه.")

    reference_value = str(globals().get("SESSION_REFERENCE_PATH", "")).strip()
    exaggeration = float(globals().get("SESSION_EXAGGERATION", 0.5))
    cfg_pace = float(globals().get("SESSION_CFG_PACE", 0.5))
    seed = int(globals().get("SESSION_SEED", 0))
    temperature = float(globals().get("SESSION_TEMPERATURE", 0.8))
    output_name = str(globals().get("SESSION_OUTPUT_NAME", "output")).strip() or "output"

    command = [
        str(VENV_PYTHON),
        "app.py",
        "--new",
        "--prompt",
        prompt_value,
        "--outputname",
        output_name,
        "--exaggeration",
        str(exaggeration),
        "--cfg-pace",
        str(cfg_pace),
        "--seed",
        str(seed),
        "--temperature",
        str(temperature),
    ]

    if reference_value:
        command.extend(["--reference", reference_value])

    print("Running command:")
    print(" ".join(command))

    result = run_cmd(command, cwd=str(PROJECT_DIR))

    stdout_text = result.stdout or ""
    stderr_text = result.stderr or ""

    if stdout_text.strip():
        print("\nCLI stdout:")
        print(stdout_text.strip())
    if stderr_text.strip():
        print("\nCLI stderr:")
        print(stderr_text.strip())

    output_path = None
    match = re.search(r"Saved audio to:\s*(.+)", stdout_text)
    if match:
        detected_path = Path(match.group(1).strip())
        output_path = detected_path if detected_path.is_absolute() else (PROJECT_DIR / detected_path)

    if output_path is None:
        outputs_dir = PROJECT_DIR / "outputs"
        candidates = sorted(outputs_dir.glob("*.wav"), key=lambda p: p.stat().st_mtime, reverse=True)
        output_path = candidates[0] if candidates else None

    if output_path is None or not output_path.exists():
        raise FileNotFoundError("لم يتم العثور على ملف صوت ناتج.")

    print(f"✅ تم التوليد: {output_path}")
    display(Audio(str(output_path), autoplay=False))
    logger.info(f"Generated output file: {output_path}")
except Exception:
    logger.exception("Cell 6 failed")
    raise

logger.info("End Cell 6")


In [ ]:
# Cell 7: (اختياري) واجهة Web عصرية كرابر للـ CLI مع رابط مؤقت
from pathlib import Path
import logging
import os
import re
import subprocess
import sys

CELL_NAME = "cell_07_optional_web_ui"
CELL_REV = "2026-04-19-r1"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

logger.info("Start Cell 7")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 7 revision: {CELL_REV}")
logger.info(f"Cell 7 revision: {CELL_REV}")


def ensure_gradio():
    try:
        import gradio as gr
        return gr
    except Exception:
        print("Installing Gradio (optional UI layer)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio>=4.44,<5"], check=True)
        import gradio as gr
        return gr


gr = ensure_gradio()

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
VENV_DIR = Path("/content/namaa-venv")
if os.name == "nt":
    default_venv_python = VENV_DIR / "Scripts" / "python.exe"
else:
    default_venv_python = VENV_DIR / "bin" / "python"

VENV_PYTHON = Path(globals().get("VENV_PYTHON", default_venv_python))

if not VENV_PYTHON.exists():
    raise FileNotFoundError("Virtualenv python not found. Run Cell 1 first.")
if not (PROJECT_DIR / "app.py").exists():
    raise FileNotFoundError("app.py not found in project directory. Run Cell 1 first.")


def run_cli_session(prompt, reference_audio, exaggeration, cfg_pace, seed, temperature, outputname):
    prompt_value = str(prompt or "").strip()
    if not prompt_value:
        return None, "❌ اكتب النص أولًا."

    output_value = str(outputname or "").strip() or "output"

    command = [
        str(VENV_PYTHON),
        "app.py",
        "--new",
        "--prompt",
        prompt_value,
        "--outputname",
        output_value,
        "--exaggeration",
        str(float(exaggeration)),
        "--cfg-pace",
        str(float(cfg_pace)),
        "--seed",
        str(int(seed)),
        "--temperature",
        str(float(temperature)),
    ]

    reference_path = str(reference_audio or "").strip()
    if reference_path:
        command.extend(["--reference", reference_path])

    result = subprocess.run(
        command,
        cwd=str(PROJECT_DIR),
        text=True,
        capture_output=True,
    )

    stdout_text = result.stdout or ""
    stderr_text = result.stderr or ""
    merged = (stdout_text + "\n" + stderr_text).strip()

    logger.info(f"CLI command: {' '.join(command)}")
    if merged:
        logger.info(merged[-4000:])

    if result.returncode != 0:
        return None, f"❌ فشل التشغيل:\n\n{merged[-2200:]}"

    output_path = None
    match = re.search(r"Saved audio to:\s*(.+)", stdout_text)
    if match:
        detected = Path(match.group(1).strip())
        output_path = detected if detected.is_absolute() else (PROJECT_DIR / detected)

    if output_path is None:
        candidates = sorted((PROJECT_DIR / "outputs").glob("*.wav"), key=lambda p: p.stat().st_mtime, reverse=True)
        output_path = candidates[0] if candidates else None

    if output_path is None or not output_path.exists():
        return None, "❌ لم يتم العثور على ملف صوت ناتج."

    status = (
        f"✅ تم التوليد بنجاح.\n\n"
        f"- output: {output_path.name}\n"
        f"- path: {output_path}\n"
        f"- reference: {'نعم' if reference_path else 'لا'}"
    )
    return str(output_path), status


custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Cairo:wght@400;600;800&display=swap');

:root {
  --n-bg: radial-gradient(1200px 520px at 12% 0%, #0f766e22, transparent 60%),
          radial-gradient(1200px 520px at 95% 100%, #0891b222, transparent 60%),
          linear-gradient(135deg, #0b1220 0%, #111827 100%);
  --n-card: #0f172ad9;
  --n-text: #e5e7eb;
  --n-accent: #14b8a6;
  --n-accent-2: #06b6d4;
}

.gradio-container {
  font-family: 'Cairo', sans-serif !important;
  background: var(--n-bg) !important;
  color: var(--n-text) !important;
}

.main-card {
  border: 1px solid #ffffff26;
  background: var(--n-card);
  border-radius: 20px;
  padding: 18px;
  box-shadow: 0 10px 40px #00000040;
}

#run-btn {
  background: linear-gradient(135deg, var(--n-accent), var(--n-accent-2)) !important;
  border: 0 !important;
  color: #001018 !important;
  font-weight: 800 !important;
}

#run-btn:hover {
  filter: brightness(1.06);
}

/* اخفاء أغلب عناصر progress الافتراضية */
.progress-text, .progress-level, .generating {
  display: none !important;
}

/* اخفاء الفوتر الافتراضي */
footer {
  display: none !important;
}
"""

with gr.Blocks(css=custom_css, title="NAMAA Studio") as demo:
    gr.Markdown(
        """
        <div class='main-card'>
          <h2 style='margin:0;'>NAMAA Studio</h2>
          <p style='margin:6px 0 0 0; opacity:.9;'>
            واجهة اختيارية مستقلة كطبقة تشغيل فوق CLI.
          </p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=5):
            prompt_box = gr.Textbox(
                label="Prompt النص",
                placeholder="اكتب النص الذي تريد تحويله...",
                lines=5,
                value="انا سبت الشغل و راجع دلوقتي علي طول.",
            )
            reference_box = gr.Audio(
                label="Reference صوتي (اختياري)",
                sources=["upload"],
                type="filepath",
            )
            with gr.Row():
                exaggeration_box = gr.Slider(0.25, 2.0, value=0.5, step=0.05, label="exaggeration")
                cfg_pace_box = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="cfg_pace")
            with gr.Row():
                seed_box = gr.Number(value=0, precision=0, label="seed")
                temperature_box = gr.Slider(0.05, 5.0, value=0.8, step=0.05, label="temperature")
            outputname_box = gr.Textbox(value="output", label="outputname")
            run_button = gr.Button("Generate Audio", elem_id="run-btn")

        with gr.Column(scale=5):
            output_audio = gr.Audio(label="Output Audio", type="filepath", autoplay=False)
            status_md = gr.Markdown("✅ جاهز للتشغيل.")

    run_button.click(
        fn=run_cli_session,
        inputs=[
            prompt_box,
            reference_box,
            exaggeration_box,
            cfg_pace_box,
            seed_box,
            temperature_box,
            outputname_box,
        ],
        outputs=[output_audio, status_md],
        show_progress="hidden",
    )

launch_result = demo.launch(
    share=True,
    inline=False,
    inbrowser=False,
    prevent_thread_lock=True,
    show_api=False,
)

share_url = None
if isinstance(launch_result, tuple) and len(launch_result) >= 3:
    share_url = launch_result[2]

if share_url:
    print(f"✅ Temporary Link: {share_url}")
else:
    print("✅ الواجهة اشتغلت. لو الرابط لم يظهر، راجع مخرجات launch بالأعلى.")

logger.info("End Cell 7")
